Import Library

In [37]:
from pathlib import Path
import torch
import pandas as pd
import sys
import os

# go up two levels: Jupyter_notebook → scripts → project_root
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
sys.path.append(project_root)


# reuse functions from your script
# from Archeived.main_predict import get_model, get_dataset, run_pred
from mst.inference.predictor import load_model, get_dataset_class, predict_batch


import warnings
warnings.simplefilter("ignore", UserWarning)

Set Device and Path

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

run_dir = Path("/home/jovyan/work/MST/runs")

# run_folder = Path("ODELIA/DinoV2ClassifierSlice_Final")
# path_run = run_dir / run_folder

##For New Model
run_folder = Path("NewModel")
checkpoint_name = "challenge_mstv3-vit_sch_CB_sub2_best.chkpt"
path_run = run_dir / run_folder / checkpoint_name

Load Model

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model("DinoV2ClassifierSlice", path_run, device)
model.to(device)
model.eval()

Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dinov3_main


DinoV2ClassifierSlice(
  (loss_func): CrossEntropyLoss()
  (auc_roc): ModuleDict(
    (train_): MulticlassAUROC()
    (val_): MulticlassAUROC()
    (test_): MulticlassAUROC()
  )
  (acc): ModuleDict(
    (train_): MulticlassAccuracy()
    (val_): MulticlassAccuracy()
    (test_): MulticlassAccuracy()
  )
  (encoder): DinoVisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (rope_embed): RopePositionEmbedding()
    (blocks): ModuleList(
      (0-11): 12 x SelfAttentionBlock(
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): SelfAttention(
          (qkv): LinearKMaskedBias(in_features=768, out_features=2304, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (norm2): Layer

In [23]:
print(model.encoder)

DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): LinearKMaskedBias(in_features=768, out_features=2304, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
    )
  )
  (norm): LayerN

In [24]:
print(model)

DinoV2ClassifierSlice(
  (loss_func): CrossEntropyLoss()
  (auc_roc): ModuleDict(
    (train_): MulticlassAUROC()
    (val_): MulticlassAUROC()
    (test_): MulticlassAUROC()
  )
  (acc): ModuleDict(
    (train_): MulticlassAccuracy()
    (val_): MulticlassAccuracy()
    (test_): MulticlassAccuracy()
  )
  (encoder): DinoVisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (rope_embed): RopePositionEmbedding()
    (blocks): ModuleList(
      (0-11): 12 x SelfAttentionBlock(
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): SelfAttention(
          (qkv): LinearKMaskedBias(in_features=768, out_features=2304, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (norm2): Layer

Load Test set

In [39]:
from mst.data.datasets.dataset_3d_odelia import ODELIA_Dataset3D

odelia_root = Path(project_root) / "mst" / "data" / "datasets" / "ODELIA_datasets"
ds_test = ODELIA_Dataset3D(path_root=odelia_root, split="test")
print(f"Loaded test samples: {len(ds_test)}")

Loaded test samples: 116


In [26]:
print(ds_test.__dict__.keys())

dict_keys(['path_root', 'path_root_data', 'split', 'sequence', 'transform', 'df', 'item_pointers'])


In [27]:
target_uid = "02F4A1FB_left"

idx = ds_test.item_pointers.index(target_uid)
sample = ds_test[idx]

In [28]:
batch = {}
for k, v in sample.items():
    if torch.is_tensor(v):
        batch[k] = v.unsqueeze(0).to(device)
    else:
        batch[k] = v


In [29]:
batch

{'uid': '02F4A1FB_left',
 'source': tensor([[[[[-0.6288, -0.6288, -0.6288,  ..., -0.6288, -0.6288, -0.6288],
            [-0.6288, -0.6288, -0.6288,  ..., -0.6288, -0.6288, -0.6288],
            [-0.6288, -0.6288, -0.6288,  ..., -0.6288, -0.6288, -0.6288],
            ...,
            [-0.5868, -0.6288, -0.6288,  ...,  1.1775,  0.9464,  0.9464],
            [-0.5868, -0.6078, -0.6498,  ...,  0.9254,  0.8624,  0.9674],
            [-0.6078, -0.6078, -0.6288,  ...,  0.8414,  0.8414,  0.9884]],
 
           [[-0.6288, -0.6288, -0.6288,  ..., -0.6288, -0.6288, -0.6288],
            [-0.6288, -0.6288, -0.6288,  ..., -0.6288, -0.6288, -0.6288],
            [-0.6288, -0.6288, -0.6288,  ..., -0.6288, -0.6288, -0.6288],
            ...,
            [-0.6078, -0.5448, -0.4818,  ...,  1.1565,  0.9044,  0.9044],
            [-0.5868, -0.5028, -0.4818,  ...,  0.9884,  0.7574,  0.7364],
            [-0.5028, -0.4397, -0.4818,  ...,  1.0514,  0.6314,  0.5264]],
 
           [[-0.6288, -0.6288, -0.628

In [30]:
batch["source"].shape

torch.Size([1, 1, 32, 224, 224])

In [31]:
pred = predict_batch(model, batch, device=device).cpu()
#probs = torch.softmax(pred, dim=-1).cpu()
pred_class = torch.argmax(pred, dim=1)

print(f"Pred is {pred}")
print(f"Predicted class is {pred_class}")

print("GT:", batch["target"].item())
print("Predicted class:", pred_class.item())
print("Probabilities:", pred.squeeze().numpy())


Pred is tensor([[0.5569, 0.2027, 0.2404]])
Predicted class is tensor([0])
GT: 0
Predicted class: 0
Probabilities: [0.5569462  0.20268923 0.24036452]


In [32]:
with torch.no_grad():
    logits = model(batch["source"].cpu())
    prob = torch.softmax(logits, dim=-1).cpu()#[0]
predicted_class = logits.argmax(dim=-1).item()

print(f"Logits: {logits}")
print(f"Probabilities: {prob}")
print(f"Predicted class from logits: {predicted_class}")

Logits: tensor([[1.0683, 0.0576, 0.2280]])
Probabilities: tensor([[0.5569, 0.2027, 0.2404]])
Predicted class from logits: 0


In [33]:
prob.detach().cpu().tolist()

[[0.5569462180137634, 0.20268923044204712, 0.24036452174186707]]

In [34]:
model.eval()

with torch.no_grad():
    logits_predict = model(
        batch["source"],
        src_key_padding_mask=batch.get("src_key_padding_mask")
    )

    # simulate perturbation step 0
    current = batch["source"].clone()

    logits_pert0 = model(
        current,
        src_key_padding_mask=batch.get("src_key_padding_mask")
    )

print(torch.allclose(logits_predict, logits_pert0, atol=1e-6))
print((logits_predict - logits_pert0).abs().max())

True
tensor(0.)


In [35]:
print(batch["source"].shape)
print(current.shape)
print(torch.equal(batch["source"], current))

torch.Size([1, 1, 32, 224, 224])
torch.Size([1, 1, 32, 224, 224])
True


In [36]:
print(batch.get("src_key_padding_mask"))

None
